# 05 — Demo: detect solar filaments in any GONG H-alpha image

Loads the 5 trained models from `weights/`, preprocesses one full-disk image, predicts filament probability,
turns it into separate filaments and shows them in colour. Works on a GPU (≈1 s) or a CPU (≈1 min).

In [ ]:
import os, sys, json, time, glob
from pathlib import Path
import numpy as np, cv2, torch, matplotlib.pyplot as plt

REPO  = Path(os.environ.get("FIL_REPO", "C:/Users/kpatel528/fil/solar-filament-seg"))   # <- repo folder
DATA  = Path(os.environ.get("FILAMENT_DATA", "C:/Users/kpatel528/fil/MAGFiLO_1.0_Kaggle_2026"))
IMAGE = sorted((DATA / "test" / "test_images").glob("*.jp*g"))[0]   # <- or any path to a GONG H-alpha image
MODELS = [f"r34_f{f}" for f in range(5)]                             # pretrained 5-fold ensemble
PARAMS = dict(thr=0.2, close_px=3, min_area=250, min_score=0.8)      # tuned on 707 out-of-fold images

sys.path.insert(0, str(REPO))
from src.preprocess import read_gray, find_disk, flatten_disk, to_uint8
from src.model import build_model
from src.infer import predict_full
from src.postprocess import instances_from_response
from src.masks import rle_to_mask
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("image:", IMAGE.name, "| device:", dev)

In [ ]:
# 1) preprocessing: find the solar disk, remove limb darkening
t0 = time.time()
raw = read_gray(str(IMAGE))
cx, cy, r = find_disk(raw)
flat, rr = flatten_disk(raw, cx, cy, r)
img = to_uint8(flat)
print(f"disk centre ({cx:.0f}, {cy:.0f}), radius {r:.0f} px")

# 2) 5-model ensemble with flip test-time augmentation
probs = []
for name in MODELS:
    cfg = json.load(open(REPO / "weights" / f"{name}.json"))
    model = build_model(cfg["encoder"], pretrained=False).to(dev)
    sd = torch.load(REPO / "weights" / f"{name}.pt", map_location=dev)
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    probs.append(predict_full(model, img, dev, tta=3))
prob = np.mean(probs, axis=0)

# 3) probability map -> separate filaments
inst = instances_from_response(prob, rr, **PARAMS)
print(f"{len(inst)} filaments found in {time.time() - t0:.1f} s")

In [ ]:
# 4) show: original | limb-flattened | probability | detected filaments
overlay = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)
rng = np.random.default_rng(1)
for rle in inst:
    m = rle_to_mask(rle)
    overlay[m] = (0.35 * overlay[m] + 0.65 * rng.integers(80, 255, 3)).astype(np.uint8)
    cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(overlay, cnts, -1, (255, 255, 0), 2)
fig, ax = plt.subplots(1, 4, figsize=(24, 6.5))
for a, (im, t, cm) in zip(ax, [(raw, f"input: {IMAGE.name}", "gray"), (img, "limb-flattened", "gray"),
                               (prob, "filament probability (5-model ensemble)", "magma"),
                               (overlay, f"{len(inst)} filaments detected", None)]):
    a.imshow(im, cmap=cm); a.set_title(t, fontsize=11); a.axis("off")
plt.tight_layout(); plt.savefig(REPO / "demo_output.png", dpi=110); plt.show()

In [ ]:
# 5) zoom on the largest filament
if inst:
    areas = [rle_to_mask(x).sum() for x in inst]
    ys, xs = np.nonzero(rle_to_mask(inst[int(np.argmax(areas))]))
    y0, y1, x0, x1 = max(ys.min() - 150, 0), ys.max() + 150, max(xs.min() - 150, 0), xs.max() + 150
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(raw[y0:y1, x0:x1], cmap="gray"); ax[0].set_title("input (zoom)")
    ax[1].imshow(overlay[y0:y1, x0:x1]); ax[1].set_title("detected filament (zoom)")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()